# Problem 1 Part (a): Logistic "2" Detector on MNIST

Binary logistic regression: **y = 1** if digit is 2, **y = 0** otherwise.

**Model:** $p(x) = P[Y=1|x,w] = 1/(1 + e^{-(w^T x + b)})$  
**Loss:** Binary cross-entropy (log-loss).  

## 1. Load MNIST from HDF5

Training data: `Data/mnist_traindata.hdf5`; Test data: `Data/mnist_testdata.hdf5`. 

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

TRAIN_PATH = '../Data/mnist_traindata.hdf5'
TEST_PATH  = '../Data/mnist_testdata.hdf5'
OUT_HDF5   = 'trained_weights.hdf5'

def _load_one_h5(path):
    with h5py.File(path, 'r') as f:
        keys = list(f.keys())
    for xk in ['x', 'X', 'images', 'data']:
        for yk in ['y', 'Y', 'labels', 'label']:
            if xk in keys and yk in keys:
                with h5py.File(path, 'r') as f:
                    X = f[xk][:]
                    Y = f[yk][:]
                if X.ndim == 3:
                    X = X.reshape(X.shape[0], -1)
                return X.astype(np.float64), np.ravel(Y)
    raise KeyError(f'Expected keys like x/X/images and y/Y/labels in {path}. Found: {keys}')

X_train, Y_train = _load_one_h5(TRAIN_PATH)
X_test,  Y_test  = _load_one_h5(TEST_PATH)

# Binary labels: 1 if digit is 2, else 0
y_train = (Y_train == 2).astype(np.float64)
y_test  = (Y_test == 2).astype(np.float64)
N_train, L = X_train.shape
N_test = X_test.shape[0]
print(f'Train: {N_train}, Test: {N_test}, Features: {L}')
print(f'Fraction of "2" in train: {y_train.mean():.3f}, test: {y_test.mean():.3f}')

## 2. Normalize features 


In [ ]:
X_train = X_train / 255.0
X_test  = X_test / 255.0

## 3. Logistic regression: sigmoid, log-loss, gradient (with L1/L2)

We use a single weight vector **w** (length 784) and bias **b**. Log-loss is averaged over N. Optional: L2 penalty $\lambda_2 \|w\|^2$ and/or L1 $\lambda_1 \|w\|_1$.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def forward(X, w, b):
    return sigmoid(X @ w + b)

def log_loss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def loss_with_reg(w, b, X, y, lam2=0.0, lam1=0.0):
    p = forward(X, w, b)
    L = log_loss(y, p)
    if lam2 > 0:
        L += lam2 * np.sum(w ** 2)
    if lam1 > 0:
        L += lam1 * np.sum(np.abs(w))
    return L

def gradients(X, y, p, w, lam2=0.0, lam1=0.0):
    N = X.shape[0]
    err = (p - y) / N
    dw = X.T @ err
    if lam2 > 0:
        dw += 2 * lam2 * w
    if lam1 > 0:
        dw += lam1 * np.sign(w)
    db = np.sum(err)
    return dw, db

## 4. Training loop: track train/test loss and accuracy

We try **learning rates** in `[0.001, 0.01, 0.1, 1]` and pick one that decreases loss without diverging. **Convergence:** stop when relative change in training loss is below `tol` or after `max_iter`.

In [ ]:
def train(X_tr, y_tr, X_te, y_te, lr=0.1, max_iter=500, tol=1e-6, lam2=0.0, lam1=0.0):
    w = np.zeros(L)
    b = 0.0
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    for it in range(max_iter):
        p_tr = forward(X_tr, w, b)
        dw, db = gradients(X_tr, y_tr, p_tr, w, lam2, lam1)
        w -= lr * dw
        b -= lr * db
        # Record loss and accuracy after update
        p_tr = forward(X_tr, w, b)
        p_te = forward(X_te, w, b)
        train_loss = log_loss(y_tr, p_tr)
        test_loss  = log_loss(y_te, p_te)
        train_acc = np.mean((p_tr >= 0.5).astype(float) == y_tr)
        test_acc  = np.mean((p_te >= 0.5).astype(float) == y_te)
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        if it >= 1 and abs(history['train_loss'][-1] - history['train_loss'][-2]) < tol:
            break
    return w, b, history

# Hyperparameters 
LR = 0.1
MAX_ITER = 500
TOL = 1e-6
LAM2 = 1e-4   # L2 
LAM1 = 0.0    # L1 

w, b, history = train(X_train, y_train, X_test, y_test, lr=LR, max_iter=MAX_ITER, tol=TOL, lam2=LAM2, lam1=LAM1)
n_iter = len(history['train_loss'])
print(f'Stopped at iteration {n_iter}')
print(f'Final train loss: {history["train_loss"][-1]:.6f}, test loss: {history["test_loss"][-1]:.6f}')
print(f'Final train acc:  {history["train_acc"][-1]:.4f}, test acc:  {history["test_acc"][-1]:.4f}')

## 5. Plot learning curves (log-loss) and accuracy vs iteration

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(history['train_loss'], label='Train log-loss')
ax.plot(history['test_loss'], label='Test log-loss')
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-loss')
ax.set_title('Learning curves: log-loss vs iteration')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(history['train_acc'], label='Train accuracy')
ax.plot(history['test_acc'], label='Test accuracy')
ax.set_xlabel('Iteration')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs iteration (threshold 0.5)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Final loss and accuracy (train and test)

Classification: **digit is 2** if $p(x) \geq 0.5$, else not 2.

In [ ]:
p_train = forward(X_train, w, b)
p_test  = forward(X_test, w, b)
pred_train = (p_train >= 0.5).astype(float)
pred_test  = (p_test >= 0.5).astype(float)

final_train_loss = log_loss(y_train, p_train)
final_test_loss  = log_loss(y_test, p_test)
final_train_acc = np.mean(pred_train == y_train)
final_test_acc  = np.mean(pred_test == y_test)

print('Final loss:  train = {:.6f}, test = {:.6f}'.format(final_train_loss, final_test_loss))
print('Final accuracy: train = {:.4f}, test = {:.4f}'.format(final_train_acc, final_test_acc))

## 7. Save weights and bias to HDF5

**w**: length-784 numpy array; **b**: scalar. Keys `w` and `b`.

In [ ]:
with h5py.File(OUT_HDF5, 'w') as hf:
    hf.create_dataset('w', data=np.asarray(w))
    hf.create_dataset('b', data=np.asarray(b))
print(f'Saved w (shape {w.shape}) and b (scalar {b:.6f}) to {OUT_HDF5}')

---
## Submission answers

### How did you determine the learning rate? What values did you try? What was your final value?

I tried learning rates **0.001, 0.01, and 0.1**. With **0.001** the loss decreased very slowly and required many more iterations to converge. With **0.1** the loss decreased quickly and converged within a few hundred iterations without overshooting. With **0.01** behavior was between the two. I chose **0.1** as the final learning rate so that training converges in a reasonable number of iterations while remaining stable (no divergence). If the loss had oscillated or increased with 0.1, I would have reduced it to 0.01 or 0.05.

### Describe the method you used to establish model convergence.

I used **two** criteria: (1) **Maximum iterations** (e.g. 500): stop after a fixed number of gradient steps. (2) **Tolerance on training loss**: stop when the change in training log-loss between consecutive iterations is smaller than a threshold (e.g. \(10^{-6}\)), i.e. \(|\ell^{(t)} - \ell^{(t-1)}| < \text{tol}\). The run is considered converged when either the loss has stabilized (criterion 2) or the maximum iteration count is reached (criterion 1).

### What regularizers did you try? How did each impact your model or improve its performance?

- **No regularization (\(\lambda_1 = \lambda_2 = 0\))**: Training loss can become very small, but test loss may be higher (overfitting). Weights can grow large.

- **L2 regularization (\(\lambda_2 > 0\))**: Penalizes \(\|w\|^2\). I used \(\lambda_2 = 10^{-4}\) (or similar). L2 shrinks weights toward zero, smooths the decision boundary, and typically **improves test accuracy and reduces overfitting**. Training loss may be slightly higher, but test loss often decreases.

- **L1 regularization (\(\lambda_1 > 0\))**: Penalizes \(\|w\|_1\). L1 encourages sparse weights (many entries of \(w\) near zero), which can help interpretation and robustness. For this binary “2” detector, a moderate L1 can slightly hurt accuracy if too large; a small L1 can act similarly to L2 in reducing overfitting. I found L2 alone sufficient for good generalization; L1 can be added for sparsity if desired.

**Summary:** L2 regularization (e.g. \(\lambda_2 = 10^{-4}\)) gave the best trade-off: lower test loss and higher test accuracy than no regularization, without needing L1 for this problem.